# Step 3: LightFM Purchase-Only Baseline — Tuning and Final Evaluation

**DS5500 Capstone — Behavior-Aware Sequential Recommendation for Next Purchase Prediction**

This notebook completes the **LightFM portion of Phase 2**: binary purchase-only interactions, WARP LightFM training, a small controlled **validation-only hyperparameter search**, comparison with the existing Popularity baseline, diagnostics for seen/unseen training items, and (only after model selection) a final test evaluation.

**Reuses, without modifying:** `artifacts/sequences_and_splits.pkl`, `artifacts/shared_evaluation_candidates.pkl`, `evaluation_utils.py`. All candidates remain in original SKU form and use the same shared evaluator.

**Does not use:** cart events, item categories, validation/test target interactions for fitting, or test metrics for model selection. **LightFM is nonsequential**; the separate purchase-only SASRec is being developed by another teammate.

**Run order:** Execute all cells with `RUN_FINAL_TEST = False`; review the tuning table and freeze the chosen model; then set `RUN_FINAL_TEST = True` and rerun **the final test cell only**. Do not retune after seeing test performance.

## 0. Setup

Run from the repository root. Install LightFM into the **same Python environment as the notebook kernel**, if needed, then restart the kernel:

```bash
python -m pip install lightfm
```

Some newer Python versions may need `lightfm-next`, which provides the same `lightfm` import. LightFM trains on CPU. Its runtime may be substantial for this catalog.

In [13]:
import os
import sys
import pickle
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
from IPython.display import display

from evaluation_utils import evaluate_model_dataset

try:
    from lightfm import LightFM
except ImportError as exc:
    raise ImportError(
        "LightFM is not installed in this notebook kernel. Install 'lightfm' "
        "(or 'lightfm-next' on newer Python), then restart the kernel."
    ) from exc

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
ARTIFACT_DIR = Path("artifacts")
SEQUENCE_PATH = ARTIFACT_DIR / "sequences_and_splits.pkl"
CANDIDATE_PATH = ARTIFACT_DIR / "shared_evaluation_candidates.pkl"
EVAL_K = 10
N_COMPONENTS = 32
LEARNING_RATE = 0.05
EPOCHS = 5  # initial checkpoint for the 32-dimension / 0.05 learning-rate run
NUM_THREADS = min(4, os.cpu_count() or 1)
BATCH_USERS = 512
RUN_FINAL_TEST = True  # leave False while selecting hyperparameters; only enable for frozen model
SAVE_MODEL = True

print(f"Python: {sys.version.split()[0]} | Threads: {NUM_THREADS}")

Python: 3.9.12 | Threads: 4


## 1. Load the existing shared artifacts

Do **not** generate new splits or resample candidates here. Candidate identifiers are original SKUs (not the shared vocabulary's +1 padded indices).

In [2]:
assert SEQUENCE_PATH.exists(), f"Missing {SEQUENCE_PATH}; run Step1_SequenceConstruction.ipynb"
assert CANDIDATE_PATH.exists(), f"Missing {CANDIDATE_PATH}; run Step2_CandidateSampling_PopularityBaseline.ipynb"

with SEQUENCE_PATH.open("rb") as f:
    data = pickle.load(f)
with CANDIDATE_PATH.open("rb") as f:
    shared_candidates = pickle.load(f)

train_sequences = data["train_sequences"]
val_targets = data["val_targets"]
test_targets = data["test_targets"]
val_eval_users = data["val_eval_users"]
test_eval_users = data["test_eval_users"]
val_candidate_sets = shared_candidates["val_candidate_sets"]
test_candidate_sets = shared_candidates["test_candidate_sets"]

assert EVAL_K == shared_candidates["eval_k"]
assert set(val_candidate_sets) == set(val_eval_users)
assert set(test_candidate_sets) == set(test_eval_users)
assert shared_candidates["val_cutoff"] == data["config"]["val_cutoff"]
assert shared_candidates["test_cutoff"] == data["config"]["test_cutoff"]
assert all(u in train_sequences for u in val_candidate_sets)
assert all(u in train_sequences for u in test_candidate_sets)
assert all(len(s) == shared_candidates["n_negatives"] + 1 for s in val_candidate_sets.values())
assert all(len(s) == shared_candidates["n_negatives"] + 1 for s in test_candidate_sets.values())
assert all(s.count(val_targets[u]) == 1 for u, s in val_candidate_sets.items())
assert all(s.count(test_targets[u]) == 1 for u, s in test_candidate_sets.items())

print(f"Training users: {len(train_sequences):,}")
print(f"Validation users: {len(val_candidate_sets):,}")
print(f"Test users (reserved): {len(test_candidate_sets):,}")
print(f"Validation cutoff: {data['config']['val_cutoff']}")
print(f"Test cutoff: {data['config']['test_cutoff']}")

Training users: 246,307
Validation users: 34,127
Test users (reserved): 35,637
Validation cutoff: 2022-11-10 00:09:15
Test cutoff: 2022-11-24 00:09:15


## 2. Build a purchase-only interaction matrix

LightFM expects a sparse **user × item** interaction matrix. We use **unique (user, purchased SKU) pairs** as implicit positive feedback. Repeated purchases of the same SKU are not duplicated in the matrix, consistent with a binary implicit-feedback baseline.

We build **compact, training-only** LightFM user/item indices. This is distinct from the shared `item_to_idx` (which reserves index 0 for SASRec padding and includes later-window SKUs). Keeping only items with training purchases avoids inadvertently predicting with random embeddings for untrained items. At evaluation, unseen-in-training candidates receive a finite low score, not a fabricated learned embedding.

**Limitation:** The saved `train_sequences` contain at most the most recent 50 purchases per user. This notebook therefore trains from those **truncated training histories**, not all historical train-period purchases. For an all-purchase LightFM baseline, Step 1 would need to save untruncated training transactions.

In [3]:
# Stable mappings for reproducibility. All SKU IDs below are original product IDs.
train_user_ids = sorted(train_sequences)
user_to_lfm = {user: i for i, user in enumerate(train_user_ids)}
train_items = sorted({sku for seq in train_sequences.values() for sku in seq})
item_to_lfm = {sku: i for i, sku in enumerate(train_items)}

row_indices, col_indices = [], []
for user, sequence in train_sequences.items():
    user_idx = user_to_lfm[user]
    for sku in set(sequence):  # binary interactions, not purchase counts
        row_indices.append(user_idx)
        col_indices.append(item_to_lfm[sku])

n_users = len(train_user_ids)
n_train_items = len(train_items)
values = np.ones(len(row_indices), dtype=np.float32)
interactions = coo_matrix(
    (values, (np.asarray(row_indices, dtype=np.int32),
              np.asarray(col_indices, dtype=np.int32))),
    shape=(n_users, n_train_items),
    dtype=np.float32,
)
assert interactions.nnz == len(row_indices)
assert np.all(interactions.data == 1)
print(f"Interaction matrix: {interactions.shape[0]:,} users × {interactions.shape[1]:,} training items")
print(f"Unique user-item purchase pairs: {interactions.nnz:,}")
print(f"Items only present outside training: {len(data['all_items']) - n_train_items:,}")
print(f"Matrix density: {interactions.nnz / (n_users * n_train_items):.8%}")

Interaction matrix: 246,307 users × 356,014 training items
Unique user-item purchase pairs: 796,880
Items only present outside training: 93,944
Matrix density: 0.00090876%


## 3. Fit the initial LightFM model (5 epochs)

Start with the original configuration. The tuning section below **continues this same model** to 10 and 20 epochs rather than retraining the first five epochs. Alternative configurations begin from the same fixed random seed.

In [4]:
model = LightFM(
    no_components=N_COMPONENTS,
    loss="warp",
    learning_rate=LEARNING_RATE,
    max_sampled=10,  # bound ranking-negative search cost for this large catalog
    random_state=RANDOM_SEED,
)
model.fit(interactions, epochs=EPOCHS, num_threads=NUM_THREADS, verbose=True)
print("LightFM training completed.")

Epoch: 100%|██████████████████████████████████████| 5/5 [00:12<00:00,  2.48s/it]

LightFM training completed.


## 4. Score the saved candidate sets

Each validation/test candidate receives a score in its **original saved order**. Products absent from the training-only LightFM vocabulary get a consistent finite low score (`-1e6`). This cold-start policy is held constant for every configuration. It does not use test labels or change candidate sets.

In [5]:
UNSEEN_ITEM_SCORE = -1e6

def score_candidate_sets(model_to_score, candidate_sets, batch_users=BATCH_USERS):
    """Return {user_id: scores in exact saved candidate order}."""
    output = {}
    users = list(candidate_sets)
    for start in range(0, len(users), batch_users):
        batch = users[start:start + batch_users]
        counts = [len(candidate_sets[u]) for u in batch]
        # All evaluation users must be in the training-user mapping.
        user_rows = np.concatenate([
            np.full(count, user_to_lfm[u], dtype=np.int32)
            for u, count in zip(batch, counts)
        ])
        item_rows = np.fromiter(
            (item_to_lfm.get(sku, -1) for u in batch for sku in candidate_sets[u]),
            dtype=np.int32,
            count=sum(counts),
        )
        flat_scores = np.full(item_rows.shape, UNSEEN_ITEM_SCORE, dtype=np.float32)
        known = item_rows >= 0
        if known.any():
            flat_scores[known] = model_to_score.predict(
                user_rows[known], item_rows[known], num_threads=NUM_THREADS
            )
        assert np.isfinite(flat_scores).all()
        offset = 0
        for user, count in zip(batch, counts):
            output[user] = flat_scores[offset:offset + count].tolist()
            offset += count
    assert set(output) == set(candidate_sets)
    return output

val_scores = score_candidate_sets(model, val_candidate_sets)
print(f"Scored {len(val_scores):,} validation users on the unchanged candidate sets.")

Scored 34,127 validation users on the unchanged candidate sets.


## 5. Evaluate initial validation checkpoint

The initial result is retained as the first row in the tuning table. We will **recompute** the selected model's validation metrics after tuning so saved outputs correspond to the selected model.

In [6]:
initial_val_summary, initial_val_results = evaluate_model_dataset(
    candidate_sets=val_candidate_sets,
    model_scores=val_scores,
    targets=val_targets,
    k=EVAL_K,
)
print("Initial (5-epoch) LightFM validation:")
display(pd.DataFrame([initial_val_summary]).round(5))

Initial (5-epoch) LightFM validation:


,n_users,HitRate@10,NDCG@10,MRR
0,34127,0.33235,0.23576,0.21935


## 6. Hyperparameter tuning (validation only)

Small, predeclared experiment set:

| Run | Factors | Learning rate | Epochs | Training method |
|---|---:|---:|---:|---|
| A | 32 | 0.05 | 5 | Existing initial fit |
| B | 32 | 0.05 | 10 | Continue A by 5 epochs |
| C | 32 | 0.05 | 20 | Continue B by 10 epochs |
| D | 64 | 0.05 | 10 | New fit |
| E | 32 | 0.01 | 10 | New fit |

All runs use WARP loss, `max_sampled=10`, seed 42, and the same purchase interaction matrix, validation users and saved candidate sets. **Choose by overall validation NDCG@10**; report HitRate@10 and MRR as additional outcomes. Keep the final test set untouched.

The model is snapshotted **whenever it becomes the best configuration**. This avoids mistakenly saving the last (rather than best) model.

In [7]:
# Keep the initial 5-epoch checkpoint as candidate A.
# Serializing only the current best avoids storing all models simultaneously.
tuning_rows = []
best_ndcg = float("-inf")
best_model_bytes = None
best_settings = None

def record_tuning_run(candidate_model, label, components, lr, epochs, existing_summary=None):
    global best_ndcg, best_model_bytes, best_settings
    if existing_summary is None:
        scores = score_candidate_sets(candidate_model, val_candidate_sets)
        summary, _ = evaluate_model_dataset(
            candidate_sets=val_candidate_sets,
            model_scores=scores,
            targets=val_targets,
            k=EVAL_K,
        )
    else:
        summary = existing_summary
    row = {
        "run": label,
        "n_components": int(components),
        "learning_rate": float(lr),
        "epochs": int(epochs),
        "HitRate@10": float(summary["HitRate@10"]),
        "NDCG@10": float(summary["NDCG@10"]),
        "MRR": float(summary["MRR"]),
    }
    tuning_rows.append(row)
    if row["NDCG@10"] > best_ndcg:
        best_ndcg = row["NDCG@10"]
        best_model_bytes = pickle.dumps(candidate_model, protocol=pickle.HIGHEST_PROTOCOL)
        best_settings = row.copy()
    print(
        f"{label}: NDCG@10={row['NDCG@10']:.5f}, "
        f"HitRate@10={row['HitRate@10']:.5f}, MRR={row['MRR']:.5f}"
    )

record_tuning_run(model, "A", 32, 0.05, 5, initial_val_summary)

# Runs B and C: continue *the original model* without resetting weights.
for label, additional_epochs, total_epochs in [("B", 5, 10), ("C", 10, 20)]:
    print(f"\nTraining run {label}: 32 factors, lr=0.05, {total_epochs} total epochs")
    model.fit_partial(
        interactions,
        epochs=additional_epochs,
        num_threads=NUM_THREADS,
        verbose=True,
    )
    record_tuning_run(model, label, 32, 0.05, total_epochs)

# Runs D and E: separate independent initializations.
for label, components, lr, epochs in [
    ("D", 64, 0.05, 10),
    ("E", 32, 0.01, 10),
]:
    print(f"\nTraining run {label}: {components} factors, lr={lr}, {epochs} epochs")
    trial_model = LightFM(
        no_components=components,
        loss="warp",
        learning_rate=lr,
        max_sampled=10,
        random_state=RANDOM_SEED,
    )
    trial_model.fit(
        interactions,
        epochs=epochs,
        num_threads=NUM_THREADS,
        verbose=True,
    )
    record_tuning_run(trial_model, label, components, lr, epochs)
    del trial_model

tuning_df = (
    pd.DataFrame(tuning_rows)
    .sort_values(["NDCG@10", "HitRate@10"], ascending=False)
    .reset_index(drop=True)
)
print("\nTuning results (sorted by validation NDCG@10):")
display(tuning_df.round(5))
print("\nSelected run:", best_settings["run"],
      "| factors:", best_settings["n_components"],
      "| lr:", best_settings["learning_rate"],
      "| epochs:", best_settings["epochs"])


A: NDCG@10=0.23576, HitRate@10=0.33235, MRR=0.21935

Training run B: 32 factors, lr=0.05, 10 total epochs


Epoch: 100%|██████████████████████████████████████| 5/5 [00:12<00:00,  2.60s/it]


B: NDCG@10=0.22679, HitRate@10=0.31758, MRR=0.21327

Training run C: 32 factors, lr=0.05, 20 total epochs


Epoch: 100%|████████████████████████████████████| 10/10 [00:25<00:00,  2.53s/it]


C: NDCG@10=0.20864, HitRate@10=0.29408, MRR=0.19794

Training run D: 64 factors, lr=0.05, 10 epochs


Epoch: 100%|████████████████████████████████████| 10/10 [00:37<00:00,  3.80s/it]


D: NDCG@10=0.22282, HitRate@10=0.30978, MRR=0.21066

Training run E: 32 factors, lr=0.01, 10 epochs


Epoch: 100%|████████████████████████████████████| 10/10 [00:23<00:00,  2.31s/it]


E: NDCG@10=0.24712, HitRate@10=0.35907, MRR=0.22475

Tuning results (sorted by validation NDCG@10):


,run,n_components,learning_rate,epochs,HitRate@10,NDCG@10,MRR
0,E,32,0.01,10,0.35907,0.24712,0.22475
1,A,32,0.05,5,0.33235,0.23576,0.21935
2,B,32,0.05,10,0.31758,0.22679,0.21327
3,D,64,0.05,10,0.30978,0.22282,0.21066
4,C,32,0.05,20,0.29408,0.20864,0.19794



Selected run: E | factors: 32 | lr: 0.01 | epochs: 10


## 7. Restore the best checkpoint and evaluate it

The selected LightFM model is restored from the best validation checkpoint. We **do not retrain** it after choosing a configuration, avoiding mismatches between tuning metrics and saved model state. Its validation scores and per-user metrics are recomputed for downstream diagnostics and saving.

In [8]:
assert best_model_bytes is not None and best_settings is not None
model = pickle.loads(best_model_bytes)
N_COMPONENTS = int(best_settings["n_components"])
LEARNING_RATE = float(best_settings["learning_rate"])
EPOCHS = int(best_settings["epochs"])

val_scores = score_candidate_sets(model, val_candidate_sets)
val_summary, val_results = evaluate_model_dataset(
    candidate_sets=val_candidate_sets,
    model_scores=val_scores,
    targets=val_targets,
    k=EVAL_K,
)
assert np.isclose(val_summary["NDCG@10"], best_settings["NDCG@10"])
assert np.isclose(val_summary["HitRate@10"], best_settings["HitRate@10"])
assert np.isclose(val_summary["MRR"], best_settings["MRR"])

print("Selected LightFM validation results:")
display(pd.DataFrame([val_summary]).round(5))

popularity_summary_path = ARTIFACT_DIR / "popularity_val_summary.csv"
if popularity_summary_path.exists():
    popularity = pd.read_csv(popularity_summary_path).iloc[0]
    comparison = pd.DataFrame([
        {"model": "Popularity", **{key: float(popularity[key]) for key in ("HitRate@10", "NDCG@10", "MRR")}},
        {"model": "LightFM (tuned; purchases only)", **{key: float(val_summary[key]) for key in ("HitRate@10", "NDCG@10", "MRR")}},
    ])
    print("Comparison on identical shared validation candidates:")
    display(comparison.round(5))
else:
    print(f"Popularity summary not present at {popularity_summary_path}; skipping comparison.")


Selected LightFM validation results:


,n_users,HitRate@10,NDCG@10,MRR
0,34127,0.35907,0.24712,0.22475


Comparison on identical shared validation candidates:


,model,HitRate@10,NDCG@10,MRR
0,Popularity,0.38823,0.25658,0.22881
1,LightFM (tuned; purchases only),0.35907,0.24712,0.22475


## 8. Seen vs. unseen training-item diagnostics

Report overall results as the main outcome. This stratification explains what fraction of validation targets are unscorable by an interaction-only model when the target SKU was never purchased during training. The **same low-score policy** is applied to all tuning experiments.

In [9]:
val_results = val_results.copy()
val_results["target_seen_in_train"] = val_results["target_item"].isin(item_to_lfm)
segment_results = (
    val_results.groupby("target_seen_in_train", dropna=False)
    .agg(n_users=("user_id", "size"), hit_rate_10=("HitRate@10", "mean"),
         ndcg_10=("NDCG@10", "mean"), mrr=("RR", "mean"))
    .reset_index()
)
segment_results["target_group"] = segment_results["target_seen_in_train"].map(
    {True: "Seen in training", False: "Unseen in training"}
)
display(segment_results[["target_group", "n_users", "hit_rate_10", "ndcg_10", "mrr"]].round(5))

assert len(val_results) == len(val_candidate_sets)
assert np.isclose(val_results["HitRate@10"].mean(), val_summary["HitRate@10"])
assert np.isclose(val_results["NDCG@10"].mean(), val_summary["NDCG@10"])
assert np.isclose(val_results["RR"].mean(), val_summary["MRR"])
print("All validation metric checks passed.")

,target_group,n_users,hit_rate_10,ndcg_10,mrr
0,Unseen in training,13337,0.00000,0.00000,0.01111
1,Seen in training,20790,0.58942,0.40566,0.36180


All validation metric checks passed.


### Optional: compare LightFM vs. Popularity on *seen* validation targets

Use the existing popularity per-user results, if present. This is a descriptive subset comparison on the exact same users, not a new evaluation population for model selection.

In [10]:
popularity_user_path = ARTIFACT_DIR / "popularity_val_user_results.csv"
if popularity_user_path.exists():
    pop_users = pd.read_csv(popularity_user_path)
    lightfm_users = val_results.copy()
    merged = lightfm_users.merge(
        pop_users[["user_id", "target_item", "HitRate@10", "NDCG@10", "RR"]],
        on=["user_id", "target_item"],
        how="inner",
        suffixes=("_lightfm", "_popularity"),
        validate="one_to_one",
    )
    assert len(merged) == len(val_results), "Popularity and LightFM validation users/targets do not match."
    rows = []
    for subset_name, part in [("All targets", merged),
                              ("Seen in training", merged[merged["target_seen_in_train"]]),
                              ("Unseen in training", merged[~merged["target_seen_in_train"]])]:
        for label, suffix in [("Popularity", "popularity"), ("LightFM", "lightfm")]:
            rows.append({
                "target_group": subset_name,
                "model": label,
                "n_users": len(part),
                "HitRate@10": part[f"HitRate@10_{suffix}"].mean(),
                "NDCG@10": part[f"NDCG@10_{suffix}"].mean(),
                "MRR": part[f"RR_{suffix}"].mean(),
            })
    display(pd.DataFrame(rows).round(5))
else:
    print("Popularity per-user validation CSV not found; subset comparison skipped.")


,target_group,model,n_users,HitRate@10,NDCG@10,MRR
0,All targets,Popularity,34127,0.38823,0.25658,0.22881
1,All targets,LightFM,34127,0.35907,0.24712,0.22475
2,Seen in training,Popularity,20790,0.63728,0.42118,0.36847
3,Seen in training,LightFM,20790,0.58942,0.40566,0.36180
4,Unseen in training,Popularity,13337,0.00000,0.00000,0.01111
5,Unseen in training,LightFM,13337,0.00000,0.00000,0.01111


## 9. Save final validation results, tuning table and selected model

Outputs follow the existing naming convention in `artifacts/`. Large model files and generated data normally stay out of Git. The **notebook itself** can be pushed once you review its output and rerun it top to bottom.

The config records the selected settings, the full tuning search and the train/candidate artifacts for reproducibility.

In [11]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
summary_path = ARTIFACT_DIR / "lightfm_val_summary.csv"
user_results_path = ARTIFACT_DIR / "lightfm_val_user_results.csv"
tuning_path = ARTIFACT_DIR / "lightfm_tuning_results.csv"

pd.DataFrame([val_summary]).to_csv(summary_path, index=False)
val_results.to_csv(user_results_path, index=False)
tuning_df.to_csv(tuning_path, index=False)

config_path = ARTIFACT_DIR / "lightfm_config.pkl"
with config_path.open("wb") as f:
    pickle.dump({
        "random_seed": RANDOM_SEED,
        "loss": "warp",
        "n_components": N_COMPONENTS,
        "learning_rate": LEARNING_RATE,
        "epochs": EPOCHS,
        "max_sampled": 10,
        "num_threads": NUM_THREADS,
        "selected_run": best_settings["run"],
        "selection_metric": "validation NDCG@10",
        "tuning_configs": tuning_rows,
        "train_source": "train_sequences (last MAX_SEQ_LEN training purchases per user)",
        "candidate_artifact": str(CANDIDATE_PATH),
        "unseen_item_score": UNSEEN_ITEM_SCORE,
        "val_cutoff": data["config"]["val_cutoff"],
        "test_cutoff": data["config"]["test_cutoff"],
    }, f)

saved_files = [summary_path, user_results_path, tuning_path, config_path]
if SAVE_MODEL:
    model_path = ARTIFACT_DIR / "lightfm_model_and_mappings.pkl"
    with model_path.open("wb") as f:
        pickle.dump({
            "model": model,
            "user_to_lfm": user_to_lfm,
            "item_to_lfm": item_to_lfm,
        }, f, protocol=pickle.HIGHEST_PROTOCOL)
    saved_files.append(model_path)

for path in saved_files:
    print(f"Saved {path} ({path.stat().st_size / 1e6:.2f} MB)")


Saved artifacts/lightfm_val_summary.csv (0.00 MB)
Saved artifacts/lightfm_val_user_results.csv (1.54 MB)
Saved artifacts/lightfm_tuning_results.csv (0.00 MB)
Saved artifacts/lightfm_config.pkl (0.00 MB)
Saved artifacts/lightfm_model_and_mappings.pkl (244.26 MB)


## 10. Final test evaluation — only after freezing model choice

**Do not run during tuning.** After the selected run and tuning table are final, change `RUN_FINAL_TEST = True` (in Setup) and execute **only this cell** (the selected model must still be loaded in your kernel). It evaluates the saved shared test candidates once, without any additional training or model selection. If you restarted your kernel, rerun the notebook from the top with `RUN_FINAL_TEST=False` before enabling this cell. Do not use the test results to retune.

In [14]:
if RUN_FINAL_TEST:
    assert best_settings is not None, "Finish validation model selection first."
    test_scores = score_candidate_sets(model, test_candidate_sets)
    test_summary, test_results = evaluate_model_dataset(
        candidate_sets=test_candidate_sets,
        model_scores=test_scores,
        targets=test_targets,
        k=EVAL_K,
    )
    print("Final LightFM test results (do not use for retuning):")
    display(pd.DataFrame([test_summary]).round(5))
    test_summary_path = ARTIFACT_DIR / "lightfm_test_summary.csv"
    test_user_results_path = ARTIFACT_DIR / "lightfm_test_user_results.csv"
    pd.DataFrame([test_summary]).to_csv(test_summary_path, index=False)
    test_results.to_csv(test_user_results_path, index=False)
    print(f"Saved {test_summary_path} and {test_user_results_path}")
else:
    print("Final test not run. Set RUN_FINAL_TEST=True only after freezing the selected model.")


Final LightFM test results (do not use for retuning):


,n_users,HitRate@10,NDCG@10,MRR
0,35637,0.28111,0.19033,0.1747


Saved artifacts/lightfm_test_summary.csv and artifacts/lightfm_test_user_results.csv


## 11. Reporting notes for Phase 2

- **Model:** LightFM binary purchase-only implicit matrix factorization with WARP loss (not sequence-aware).
- **Tuning:** Five predeclared training configurations; selection on overall validation NDCG@10 only. Record chosen factors, learning rate, epochs and full tuning table.
- **Fair comparison:** Shared temporal splits, validation/test users, candidates and `evaluation_utils.py`; compare to Popularity, then purchase-only SASRec when available.
- **Cold-start:** Missing-in-training SKUs receive a fixed low score; stratify by training-item coverage and interpret overall performance accordingly.
- **Scope:** `train_sequences` are last-50 purchase histories, not full training purchase matrices. Metrics use 101 sampled candidates/user, not full-catalog retrieval.
- **Test:** One final evaluation after model selection. Store per-user and summary results.

### LightFM Baseline Results and Discussion

We implemented a purchase-only LightFM baseline using binary user-item interactions and WARP ranking loss. Five configurations were evaluated using validation NDCG@10, and the best model used 32 latent dimensions, a learning rate of 0.01, and 10 training epochs.

The selected model achieved a validation HitRate@10 of 0.3591, NDCG@10 of 0.2471, and MRR of 0.2248. On the held-out test set, performance decreased to 0.2811, 0.1903, and 0.1747, respectively.

LightFM did not outperform the global Popularity baseline on validation. One important limitation was the large proportion of target products unseen during training (39.1% of validation targets), which the purchase-only model could not effectively recommend. Even among previously seen products, Popularity slightly outperformed LightFM.

These findings suggest that personalized recommendations based on purchase interactions alone may be limited by sparse purchase histories and item cold-start effects. The results provide a benchmark for evaluating whether incorporating cart behavior improves next-purchase prediction.